In [1]:
import sys
from petsc4py import PETSc
import time  

sys.path.append("../Solver")

from Module import *

import kron_test


--------------------------------------------------------------------------
but there are no active ports detected (or Open MPI was unable to use
them).  This is most certainly not what you wanted.  Check your
cables, subnet manager configuration, etc.  The openib BTL will be
ignored for this job.

  Local host: terra
--------------------------------------------------------------------------


In [11]:
def kronV5(A,B,nonzeros):
    ra,ca = A.getSize()
    rb,cb = B.getSize()

    C_par = PETSc.Mat().createAIJ([ra*rb,ca*cb],comm = PETSc.COMM_WORLD,nnz = nonzeros)
    ownershipC = C_par.getOwnershipRange()
    C_range = range(ownershipC[0],ownershipC[1])

    seq_A = getLocal(A)
    seq_B = getLocal(B)

    C_seq = seq_A.kron(seq_B)
    
    
    for i in C_range:
        indices,values = C_seq.getRow(i)
        C_par.setValues(i,indices,values)
    
    C_par.assemble()

    seq_A.destroy()
    seq_B.destroy()
    C_seq.destroy()

    return C_par


def kronV6(A,B,nonzeros):
    ra,ca = A.getSize()
    rb,cb = B.getSize()

    C_par = PETSc.Mat().createAIJ([ra*rb,ca*cb],comm = PETSc.COMM_WORLD,nnz = nonzeros)
    ownershipC = C_par.getOwnershipRange()
    C_range = range(ownershipC[0],ownershipC[1])

    seq_A = getLocal(A)
    seq_B = getLocal(B)
    

    C_seq = seq_A.kron(seq_B)

    row_pointers = [0]
    column_indices = []
    value_list = []



    for i in range(ra*rb):
        indices,values = C_seq.getRow(i)
        row_pointers.append(row_pointers[-1] + len(indices))
        column_indices.append(indices)
        value_list.append(values)
    
    column_indices = np.concatenate(column_indices)
    value_list = np.concatenate(value_list)
    
    
    C_par.setValuesCSR(row_pointers,column_indices,value_list)
    

    C_par.assemble()
        
    seq_A.destroy()
    seq_B.destroy()

    return C_par





In [20]:
A = PETSc.Mat().createAIJ([250,250])
for i in range(250):
    for j in range(250):
        if i == j+1:
            A.setValue(i,j,1)
        elif j == i+1:
            A.setValue(i,j,1)
A.assemble()

B = PETSc.Mat().createAIJ([200,200])
for i in range(200):
    for j in range(200):
        B.setValue(i,j,1)

        
B.assemble()





In [21]:
start = time.time()
C1 = kronV4(A,B,400)
end = time.time()
print(end-start)

3.9488394260406494


In [22]:
start = time.time()
C2 = kronV5(A,B,400)
end = time.time()
print(end-start)

2.307741641998291
